In [60]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import igraph as ig
import random
import time

from bayes_opt import BayesianOptimization
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score

In [61]:
BITCOIN_DATA_PATH = "../data/bitcoin/soc-sign-bitcoinalpha.csv"
LASFTM_EDGES_PATH = "../data/lasftm_asia/lastfm_asia_edges.csv"
EMAIL_EDGES_PATH = "../data/email/email-Eu-core.txt"
EMAIL_LABELS_PATH = "../data/email/email-Eu-core-department-labels.txt"
DBLP_EDGES_PATH = "../data/DBLP/com-dblp-ungraph.txt"
DBLP_LABELS_PATH = "../data/DBLP/com-dblp-all-cmty.txt"
JULIA_GRAPH_1_PATH = "../data/JuliaABCDGenerator/Graph1/network.dat"
JULIA_GRAPH_2_PATH = "../data/JuliaABCDGenerator/Graph2/network.dat"
JULIA_GRAPH_3_PATH = "../data/JuliaABCDGenerator/Graph3/network.dat"
PLOT_OPTIONS = {"node_size": 10, "with_labels": False, "width": 0.15}
FIGURE_SIZE = (10, 10)
SEED = 151936
NO_TOP_NODES = 10


plt.style.use("default")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"

In [72]:
def load_graph_from_edgelist(path, separator=" "):
    edges = []

    with open(path, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            
            if separator == " ":
                u, v = map(int, line.strip().split())
            else:
                u, v = map(int, line.strip().split(separator))
            edges.append((u, v))

    g = ig.Graph.TupleList(edges, directed=False)

    return g

def load_graph_with_labels(edge_path, label_path):
    g = load_graph_from_edgelist(edge_path)

    labels = {}
    with open(label_path, "r") as f:
        for line in f:
            node, label = map(int, line.strip().split())
            labels[node] = label

    ground_truth = [labels.get(v.index, -1) for v in g.vs]
    return g, ground_truth

def load_bitcoin_graph(path):
    edges = []
    weights = []

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 2:
                u = int(parts[0])
                v = int(parts[1])
                w = float(parts[2])
                edges.append((u, v))
                weights.append(w)

    g = ig.Graph(edges=edges, directed=True)
    g.es["weight"] = weights

    g.simplify(combine_edges="sum")
    g = g.as_undirected(combine_edges="sum")
    return g

def load_dblp_graph(path):
    edges = []

    with open(path, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            u, v = map(int, line.strip().split())
            edges.append((u, v))

    g = ig.Graph.TupleList(edges, directed=False)

    return g

def load_dblp_ground_truth(path, graph):
    node_to_comm = {}

    with open(path, "r") as f:
        for comm_id, line in enumerate(f):
            nodes = line.strip().split()
            for node in nodes:
                node_to_comm[node] = comm_id

    gt = []
    for v in graph.vs:
        node_id = v["name"]
        gt.append(node_to_comm.get(str(node_id), -1))

    return gt

def load_ground_truth_communities(path, graph):
    labels = {}

    with open(path, "r") as f:
        for line in f:
            node, comm = map(int, line.strip().split())
            labels[str(node)] = comm

    gt = []
    for v in graph.vs:
        node_id = v["name"]
        gt.append(labels.get(str(node_id), -1))

    return gt

def run_leiden(g, resolution=1.0, seed=SEED):
    random.seed(seed)
    return g.community_leiden(objective_function="modularity", resolution=resolution)

def run_louvain(g):
    return g.community_multilevel()

def run_label_propagation(g):
    return g.community_label_propagation()

def run_infomap(g):
    return g.community_infomap()

def compute_modularity(g, clustering):
    return g.modularity(clustering.membership)

def clustering_to_labels(clustering):
    return np.array(clustering.membership)

def compare_algorithms_modularity(g, algorithms):
    results = {}

    for name, algo in algorithms.items():
        clustering = algo(g)
        modularity = compute_modularity(g, clustering)

        results[name] = {
            "clustering": clustering,
            "modularity": modularity
        }

    return results

def compare_with_ground_truth(g, ground_truth, algorithms):
    results = {}

    for name, algo in algorithms.items():
        start = time.time()

        clustering = algo(g)

        end = time.time()
        runtime = end - start

        pred = clustering.membership

        modularity = g.modularity(pred)
        ami = adjusted_mutual_info_score(ground_truth, pred)
        ari = adjusted_rand_score(ground_truth, pred)

        deterministic, _ = check_determinism(g, algo)

        results[name] = {
            "modularity": modularity,
            "AMI": ami,
            "ARI": ari,
            "time": runtime,
            "deterministic": deterministic
        }

    return results


def evaluate_on_synthetic_with_gt(graph_paths, algorithms):
    all_results = {}

    for path in graph_paths:
        g = load_graph_from_edgelist(path)

        gt_path = path.replace("network.dat", "community.dat")
        ground_truth = load_ground_truth_communities(gt_path, g)

        results = {}

        for name, algo in algorithms.items():
            start = time.time()

            clustering = algo(g)

            end = time.time()
            runtime = end - start

            pred = clustering.membership

            modularity = g.modularity(pred)
            ami = adjusted_mutual_info_score(ground_truth, pred)
            ari = adjusted_rand_score(ground_truth, pred)

            deterministic, _ = check_determinism(g, algo)

            results[name] = {
                "modularity": modularity,
                "AMI": ami,
                "ARI": ari,
                "time": runtime,
                "deterministic": deterministic
            }

        all_results[path] = results

    return all_results

def check_determinism(g, algorithm, runs=5):
    partitions = []

    for _ in range(runs):
        clustering = algorithm(g)
        partitions.append(tuple(clustering.membership))

    unique = len(set(partitions))
    return unique == 1, unique

def leiden_objective(resolution, g):
    clustering = g.community_leiden(
        objective_function="modularity",
        resolution=resolution
    )
    return g.modularity(clustering.membership)


def optimize_leiden_resolution(g):
    optimizer = BayesianOptimization(
        f=lambda resolution: leiden_objective(resolution, g),
        pbounds={"resolution": (0.1, 2.0)},
        random_state=SEED,
    )

    optimizer.maximize(init_points=5, n_iter=10)
    return optimizer.max

In [73]:
algorithms = {
    "Leiden": lambda g: run_leiden(g, resolution=1.0),
    "Louvain": run_louvain,
    "LabelProp": run_label_propagation,
    "Infomap": run_infomap
}

### Task 1

In [74]:
g1 = load_bitcoin_graph(BITCOIN_DATA_PATH)
g2 = load_graph_from_edgelist(LASFTM_EDGES_PATH, separator=",")

res1 = compare_algorithms_modularity(g1, algorithms)
res2 = compare_algorithms_modularity(g2, algorithms)

print("Bitcoin Modularity:")
for algo, result in res1.items():
    print(f"{algo}: {result['modularity']:.4f}")

print("\nLASFTM Modularity:")
for algo, result in res2.items():
    print(f"{algo}: {result['modularity']:.4f}")

Bitcoin Modularity:
Leiden: 0.4783
Louvain: 0.4676
LabelProp: 0.0103
Infomap: 0.3993

LASFTM Modularity:
Leiden: 0.8157
Louvain: 0.8145
LabelProp: 0.7907
Infomap: 0.7307


### Task 2a

In [76]:
g1, gt1 = load_graph_with_labels(EMAIL_EDGES_PATH, EMAIL_LABELS_PATH)
g2 = load_dblp_graph(DBLP_EDGES_PATH)
gt2 = load_dblp_ground_truth(DBLP_LABELS_PATH, g2)
results1 = compare_with_ground_truth(g1, gt1, algorithms)
results2 = compare_with_ground_truth(g2, gt2, algorithms)

print("\nEmail Ground Truth Comparison:")
for algo, result in results1.items():
    print(f"{algo}: Modularity={result['modularity']:.4f}, AMI={result['AMI']:.4f}, ARI={result['ARI']:.4f}, Time={result['time']:.4f}s, Deterministic={result['deterministic']}")

print("\nDBLP Ground Truth Comparison:")
for algo, result in results2.items():
    print(f"{algo}: Modularity={result['modularity']:.4f}, AMI={result['AMI']:.4f}, ARI={result['ARI']:.4f}, Time={result['time']:.4f}s, Deterministic={result['deterministic']}")


Email Ground Truth Comparison:
Leiden: Modularity=0.4399, AMI=0.5653, ARI=0.3302, Time=0.0044s, Deterministic=True
Louvain: Modularity=0.4389, AMI=0.5576, ARI=0.3271, Time=0.0154s, Deterministic=False
LabelProp: Modularity=0.0019, AMI=-0.0039, ARI=-0.0009, Time=0.0095s, Deterministic=False
Infomap: Modularity=0.4218, AMI=0.6032, ARI=0.3611, Time=0.4078s, Deterministic=False

DBLP Ground Truth Comparison:
Leiden: Modularity=0.8302, AMI=0.2436, ARI=0.0082, Time=2.5932s, Deterministic=True
Louvain: Modularity=0.8212, AMI=0.2317, ARI=0.0075, Time=6.1670s, Deterministic=False
LabelProp: Modularity=0.6822, AMI=0.2322, ARI=-0.0231, Time=101.2474s, Deterministic=False
Infomap: Modularity=0.7312, AMI=0.2494, ARI=0.0009, Time=190.8623s, Deterministic=False


### Task 2b

In [75]:
graphs = [JULIA_GRAPH_1_PATH, JULIA_GRAPH_2_PATH, JULIA_GRAPH_3_PATH]
synthetic_results = evaluate_on_synthetic_with_gt(graphs, algorithms)

print("\nSynthetic Graph Results:")

for path, res in synthetic_results.items():
    print(f"\nGraph: {path}")
    for algo, result in res.items():
        print(
            f"{algo}: "
            f"Modularity={result['modularity']:.4f}, "
            f"AMI={result['AMI']:.4f}, "
            f"ARI={result['ARI']:.4f}, "
            f"Time={result['time']:.4f}s, "
            f"Deterministic={result['deterministic']}"
        )


Synthetic Graph Results:

Graph: ../data/JuliaABCDGenerator/Graph1/network.dat
Leiden: Modularity=0.6727, AMI=0.9292, ARI=0.9420, Time=0.0040s, Deterministic=True
Louvain: Modularity=0.6722, AMI=0.9279, ARI=0.9379, Time=0.0040s, Deterministic=False
LabelProp: Modularity=0.6005, AMI=0.8823, ARI=0.6349, Time=0.0000s, Deterministic=False
Infomap: Modularity=0.6692, AMI=0.9519, ARI=0.9726, Time=0.1970s, Deterministic=False

Graph: ../data/JuliaABCDGenerator/Graph2/network.dat
Leiden: Modularity=0.7139, AMI=0.3455, ARI=0.1528, Time=0.0060s, Deterministic=True
Louvain: Modularity=0.7001, AMI=0.3024, ARI=0.1247, Time=0.0257s, Deterministic=False
LabelProp: Modularity=0.6676, AMI=0.2955, ARI=0.0780, Time=0.0155s, Deterministic=False
Infomap: Modularity=0.6096, AMI=0.2649, ARI=0.0200, Time=0.5443s, Deterministic=False

Graph: ../data/JuliaABCDGenerator/Graph3/network.dat
Leiden: Modularity=0.5782, AMI=0.9293, ARI=0.9689, Time=0.0275s, Deterministic=True
Louvain: Modularity=0.5779, AMI=0.9258, 

### Task 2c

In [42]:
best = optimize_leiden_resolution(g1)
print(best)


|   iter    |  target   | resolu... |
-------------------------------------
| 1         | 0.4790325 | 0.8116262 |
| 2         | 0.4387768 | 1.9063571 |
| 3         | 0.4630175 | 1.4907884 |
| 4         | 0.4767684 | 1.2374511 |
| 5         | 0.1545682 | 0.3964354 |
| 6         | 0.4792711 | 0.8116426 |
| 7         | 0.4712970 | 0.8334064 |
| 8         | 0.4476653 | 1.6940855 |
| 9         | 0.4807847 | 1.0711919 |
| 10        | 0.0019808 | 0.1       |
| 11        | 0.4679947 | 0.6743769 |
| 12        | 0.4354826 | 2.0       |
| 13        | 0.4724939 | 1.3604109 |
| 14        | 0.4692174 | 1.1437892 |
| 15        | 0.4750743 | 0.7411723 |
{'target': np.float64(0.4807847640444175), 'params': {'resolution': np.float64(1.0711919867027087)}}


In [43]:
best = optimize_leiden_resolution(g2)
print(best)


|   iter    |  target   | resolu... |
-------------------------------------
| 1         | 0.4370384 | 0.8116262 |
| 2         | 0.4125326 | 1.9063571 |
| 3         | 0.4266671 | 1.4907884 |
| 4         | 0.4327667 | 1.2374511 |
| 5         | 0.1071157 | 0.3964354 |
| 6         | 0.4366812 | 0.8116431 |
| 7         | 0.4366523 | 0.7987624 |
| 8         | 0.4166092 | 1.6988210 |
| 9         | 0.4400123 | 1.0525039 |
| 10        | 0.4114162 | 2.0       |
| 11        | 0.4309945 | 1.3634915 |
| 12        | 0.4340024 | 0.9566414 |
| 13        | 0.4340607 | 1.1360030 |
| 14        | 0.4213549 | 1.5887510 |
| 15        | 0.4379431 | 1.0184651 |
{'target': np.float64(0.4400123340809178), 'params': {'resolution': np.float64(1.052503908298012)}}
